<a href="https://colab.research.google.com/github/awa-easymoneysniper/OSU-Mania-Auto-Beatmap-Generator/blob/main/mania_beatmap_feature_extractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# =========================
# 1. 配置与依赖
# =========================

import json
import math
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

BEATMAP_EVENT_DIR = Path(
    "/content/drive/MyDrive/osu_mania_dataset/beatmap_events_4k"
)
BEATMAP_INDEX_PATH = BEATMAP_EVENT_DIR / "beatmap_index.json"

# 如果你的音频 embedding 目录不同，只改这里
AUDIO_EMBEDDING_DIR = Path(
    "/content/drive/MyDrive/osu_mania_dataset/audio_embeddings"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/osu_mania_dataset/beatmap_embeddings"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

AUDIO_DIM = 768
D_MODEL = 256
NUM_HEADS = 8
NUM_LAYERS = 4
FF_DIM = 1024

MAX_SEQ_LEN = 1024
BATCH_SIZE = 32
EPOCHS = 40
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
TEMPERATURE = 0.07

# 辅助任务：从 event 学习预测 star rating
LAMBDA_STAR = 0.10

Device: cuda


In [ ]:
# =========================
# 2. 读取 beatmap index 与所有 shard
# =========================

with open(BEATMAP_INDEX_PATH, "r", encoding="utf-8") as f:
    beatmap_index = json.load(f)

audio_to_samples = beatmap_index["audio_to_samples"]

shard_paths = sorted(BEATMAP_EVENT_DIR.glob("shard_*.pt"))
print("Beatmap shards:", len(shard_paths))

# 约 2 万张谱面，int16 event 数据通常可直接加载到 Colab RAM，
# 可避免训练时从 Google Drive 反复随机读取 shard。
beatmap_shards = {}

for shard_path in tqdm(shard_paths, desc="Loading beatmap shards"):
    shard = torch.load(shard_path, map_location="cpu", weights_only=False)
    beatmap_shards[shard["shard_id"]] = shard

print("Loaded beatmap shards:", len(beatmap_shards))

Beatmap shards: 38


Loading beatmap shards:   0%|          | 0/38 [00:00<?, ?it/s]

Loaded beatmap shards: 38


original

In [ ]:
# =========================
# 3. 读取并验证 audio embeddings
# =========================

# 使用 MERT 最后 4 层的平均。
# 这是比只选最后一层更稳定的起始选择。
NUM_MERT_LAYERS_TO_AVERAGE = 4


def find_audio_embedding_path(audio_id):
    candidates = [
        AUDIO_EMBEDDING_DIR / f"{audio_id}.npz",
        AUDIO_EMBEDDING_DIR / f"{audio_id}.pt",
        AUDIO_EMBEDDING_DIR / f"{audio_id}.pth",
        AUDIO_EMBEDDING_DIR / f"{audio_id}.npy",
    ]

    for path in candidates:
        if path.exists():
            return path

    return None


def unpack_audio_embedding(obj):
    """
    最终返回全局音频 embedding: [768]

    对 MERT .npz:
    pooled_per_chunk: [num_layers, num_chunks, 768]
    先平均最后 4 个 MERT layer，再平均所有 10 秒 chunk。
    """
    if isinstance(obj, np.lib.npyio.NpzFile):
        if "pooled_per_chunk" not in obj:
            raise ValueError(
                f"Missing pooled_per_chunk. Keys: {obj.files}"
            )

        x = torch.from_numpy(obj["pooled_per_chunk"]).float()

        if x.ndim != 3:
            raise ValueError(
                f"Expected [layers, chunks, dim], got {tuple(x.shape)}"
            )

        # [layers, chunks, 768] -> [768]
        x = x[-NUM_MERT_LAYERS_TO_AVERAGE:].mean(dim=(0, 1))
        return F.normalize(x, dim=0)

    if isinstance(obj, dict):
        for key in ["embedding", "embeddings", "audio_embedding", "features"]:
            if key in obj:
                obj = obj[key]
                break
        else:
            raise ValueError(f"Unknown keys: {obj.keys()}")

    if isinstance(obj, np.ndarray):
        obj = torch.from_numpy(obj)

    if not torch.is_tensor(obj):
        raise TypeError(f"Expected Tensor or ndarray, got {type(obj)}")

    x = obj.detach().float().cpu()

    if x.ndim == 2:
        x = x.mean(dim=0)

    if x.ndim != 1 or x.numel() != AUDIO_DIM:
        raise ValueError(
            f"Expected [{AUDIO_DIM}], got {tuple(x.shape)}"
        )

    return F.normalize(x, dim=0)


# 512 train + 64 val + 64 test
DEBUG_NUM_AUDIO = len(audio_to_samples)

candidate_audio_ids = sorted(audio_to_samples.keys())

debug_rng = random.Random(SEED)
debug_rng.shuffle(candidate_audio_ids)

candidate_audio_ids = candidate_audio_ids[:DEBUG_NUM_AUDIO]

audio_embeddings = {}
missing_audio_embeddings = []
invalid_audio_embeddings = []

for audio_id in tqdm(
    candidate_audio_ids,
    desc="Loading debug audio embeddings",
):
    embedding_path = find_audio_embedding_path(audio_id)

    if embedding_path is None:
        missing_audio_embeddings.append(audio_id)
        continue

    try:
        if embedding_path.suffix == ".npz":
            raw = np.load(embedding_path)
        elif embedding_path.suffix == ".npy":
            raw = np.load(embedding_path)
        else:
            raw = torch.load(
                embedding_path,
                map_location="cpu",
                weights_only=False,
            )

        audio_embeddings[audio_id] = unpack_audio_embedding(raw)

    except Exception as e:
        invalid_audio_embeddings.append({
            "audio_id": audio_id,
            "path": str(embedding_path),
            "error": repr(e),
        })

valid_audio_ids = sorted(audio_embeddings.keys())

print("Debug 候选音频数:", len(candidate_audio_ids))
print("可用音频 embedding 数:", len(valid_audio_ids))
print("缺失 embedding 数:", len(missing_audio_embeddings))
print("无效 embedding 数:", len(invalid_audio_embeddings))

Loading debug audio embeddings:   0%|          | 0/6998 [00:00<?, ?it/s]

KeyboardInterrupt: 

多线程

In [ ]:
# =========================
# 3. 读取并验证 audio embeddings
# =========================

from concurrent.futures import ThreadPoolExecutor, as_completed


# 使用 MERT 最后 4 层的平均。
# 这是比只选最后一层更稳定的起始选择。
NUM_MERT_LAYERS_TO_AVERAGE = 4

# Google Drive 小文件并行读取线程数。
# 若频繁读取报错或速度反而变慢，改为 4。
NUM_IO_WORKERS = 8


def find_audio_embedding_path(audio_id):
    candidates = [
        AUDIO_EMBEDDING_DIR / f"{audio_id}.npz",
        AUDIO_EMBEDDING_DIR / f"{audio_id}.pt",
        AUDIO_EMBEDDING_DIR / f"{audio_id}.pth",
        AUDIO_EMBEDDING_DIR / f"{audio_id}.npy",
    ]

    for path in candidates:
        if path.exists():
            return path

    return None


def unpack_audio_embedding(obj):
    """
    最终返回全局音频 embedding: [768]

    对 MERT .npz:
    pooled_per_chunk: [num_layers, num_chunks, 768]
    先平均最后 4 个 MERT layer，再平均所有 10 秒 chunk。
    """
    if isinstance(obj, np.lib.npyio.NpzFile):
        if "pooled_per_chunk" not in obj:
            raise ValueError(
                f"Missing pooled_per_chunk. Keys: {obj.files}"
            )

        x = torch.from_numpy(obj["pooled_per_chunk"]).float()

        if x.ndim != 3:
            raise ValueError(
                f"Expected [layers, chunks, dim], got {tuple(x.shape)}"
            )

        # [layers, chunks, 768] -> [768]
        x = x[-NUM_MERT_LAYERS_TO_AVERAGE:].mean(dim=(0, 1))
        return F.normalize(x, dim=0)

    if isinstance(obj, dict):
        for key in ["embedding", "embeddings", "audio_embedding", "features"]:
            if key in obj:
                obj = obj[key]
                break
        else:
            raise ValueError(f"Unknown keys: {obj.keys()}")

    if isinstance(obj, np.ndarray):
        obj = torch.from_numpy(obj)

    if not torch.is_tensor(obj):
        raise TypeError(f"Expected Tensor or ndarray, got {type(obj)}")

    x = obj.detach().float().cpu()

    if x.ndim == 2:
        x = x.mean(dim=0)

    if x.ndim != 1 or x.numel() != AUDIO_DIM:
        raise ValueError(
            f"Expected [{AUDIO_DIM}], got {tuple(x.shape)}"
        )

    return F.normalize(x, dim=0)


# 512 train + 64 val + 64 test
# 正式训练时取全部有 4K 谱面的 audio_id
DEBUG_NUM_AUDIO = len(audio_to_samples)

candidate_audio_ids = sorted(audio_to_samples.keys())

debug_rng = random.Random(SEED)
debug_rng.shuffle(candidate_audio_ids)

candidate_audio_ids = candidate_audio_ids[:DEBUG_NUM_AUDIO]


def load_one_audio_embedding(audio_id):
    """
    单线程任务：读取并转换一首歌的 embedding。
    返回结果而不是直接写全局 dict，避免多线程写入共享对象。
    """
    embedding_path = find_audio_embedding_path(audio_id)

    if embedding_path is None:
        return {
            "audio_id": audio_id,
            "embedding": None,
            "missing": True,
            "error": None,
            "path": None,
        }

    try:
        if embedding_path.suffix == ".npz":
            # 用 with 自动关闭 npz 文件，避免同时打开过多文件
            with np.load(embedding_path) as raw:
                embedding = unpack_audio_embedding(raw)

        elif embedding_path.suffix == ".npy":
            raw = np.load(embedding_path)
            embedding = unpack_audio_embedding(raw)

        else:
            raw = torch.load(
                embedding_path,
                map_location="cpu",
                weights_only=False,
            )
            embedding = unpack_audio_embedding(raw)

        return {
            "audio_id": audio_id,
            "embedding": embedding,
            "missing": False,
            "error": None,
            "path": str(embedding_path),
        }

    except Exception as e:
        return {
            "audio_id": audio_id,
            "embedding": None,
            "missing": False,
            "error": repr(e),
            "path": str(embedding_path),
        }


audio_embeddings = {}
missing_audio_embeddings = []
invalid_audio_embeddings = []

with ThreadPoolExecutor(max_workers=NUM_IO_WORKERS) as executor:
    futures = [
        executor.submit(load_one_audio_embedding, audio_id)
        for audio_id in candidate_audio_ids
    ]

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc=(
            f"Loading debug audio embeddings "
            f"({NUM_IO_WORKERS} threads)"
        ),
    ):
        result = future.result()

        if result["missing"]:
            missing_audio_embeddings.append(result["audio_id"])

        elif result["error"] is not None:
            invalid_audio_embeddings.append({
                "audio_id": result["audio_id"],
                "path": result["path"],
                "error": result["error"],
            })

        else:
            audio_embeddings[
                result["audio_id"]
            ] = result["embedding"]

valid_audio_ids = sorted(audio_embeddings.keys())

print("Debug 候选音频数:", len(candidate_audio_ids))
print("可用音频 embedding 数:", len(valid_audio_ids))
print("缺失 embedding 数:", len(missing_audio_embeddings))
print("无效 embedding 数:", len(invalid_audio_embeddings))

Loading debug audio embeddings (8 threads):   0%|          | 0/6998 [00:00<?, ?it/s]

Debug 候选音频数: 6998
可用音频 embedding 数: 6998
缺失 embedding 数: 0
无效 embedding 数: 0


保存整体文件

In [ ]:
# =========================
# 保存全局 audio embedding 缓存
# =========================

AUDIO_GLOBAL_CACHE = (
    AUDIO_EMBEDDING_DIR / "mert_global_embeddings_4k.pt"
)

audio_cache = {
    "audio_ids": valid_audio_ids,
    "embeddings": torch.stack([
        audio_embeddings[audio_id]
        for audio_id in valid_audio_ids
    ]),  # [num_audio, 768]
    "missing_audio_embeddings": missing_audio_embeddings,
    "invalid_audio_embeddings": invalid_audio_embeddings,
    "mert_pooling": (
        f"mean(last_{NUM_MERT_LAYERS_TO_AVERAGE}_layers, "
        "all_10s_chunks)"
    ),
}

torch.save(audio_cache, AUDIO_GLOBAL_CACHE)

print("Saved:", AUDIO_GLOBAL_CACHE)
print("Tensor shape:", audio_cache["embeddings"].shape)

Saved: /content/drive/MyDrive/osu_mania_dataset/audio_embeddings/mert_global_embeddings_4k.pt
Tensor shape: torch.Size([6998, 768])


读取整体文件

In [ ]:
# =========================
# 3. 从缓存读取 audio embeddings
# =========================

AUDIO_GLOBAL_CACHE = (
    AUDIO_EMBEDDING_DIR / "mert_global_embeddings_4k.pt"
)

audio_cache = torch.load(
    AUDIO_GLOBAL_CACHE,
    map_location="cpu",
    weights_only=False,
)

valid_audio_ids = audio_cache["audio_ids"]

audio_embeddings = {
    audio_id: embedding
    for audio_id, embedding in zip(
        audio_cache["audio_ids"],
        audio_cache["embeddings"],
    )
}

missing_audio_embeddings = audio_cache.get(
    "missing_audio_embeddings",
    [],
)

invalid_audio_embeddings = audio_cache.get(
    "invalid_audio_embeddings",
    [],
)

print("Loaded cached audio embeddings:", len(valid_audio_ids))
print("Tensor shape:", audio_cache["embeddings"].shape)

Loaded cached audio embeddings: 6998
Tensor shape: torch.Size([6998, 768])


In [ ]:
# =========================
# 4. 按 audio_id 划分 train / val / test
# =========================

rng = random.Random(SEED)
rng.shuffle(valid_audio_ids)

n_total = len(valid_audio_ids)
n_train = int(n_total * 0.80)
n_val = int(n_total * 0.10)

train_audio_ids = valid_audio_ids[:n_train]
val_audio_ids = valid_audio_ids[n_train:n_train + n_val]
test_audio_ids = valid_audio_ids[n_train + n_val:]

# train_audio_ids = train_audio_ids[:512]
# val_audio_ids = val_audio_ids[:128]

# EPOCHS = 40
# BATCH_SIZE = 32

print(
    f"Train: {len(train_audio_ids)}, "
    f"Val: {len(val_audio_ids)}, "
    f"Test: {len(test_audio_ids)}"
)

split_info = {
    "seed": SEED,
    "train_audio_ids": train_audio_ids,
    "val_audio_ids": val_audio_ids,
    "test_audio_ids": test_audio_ids,
}

with open(OUTPUT_DIR / "audio_splits.json", "w", encoding="utf-8") as f:
    json.dump(split_info, f, ensure_ascii=False, indent=2)

Train: 5598, Val: 699, Test: 701


In [ ]:
# =========================
# 5. Dataset
#
# 一个 batch 内每个 audio_id 只出现一次。
# 同一首歌的多个 beatmap 每次随机抽一张。
# 这样不会将同一首歌的不同难度错误地当作负样本。
# =========================

class AudioGroupedBeatmapDataset(Dataset):
    def __init__(
        self,
        audio_ids,
        audio_to_samples,
        beatmap_shards,
        audio_embeddings,
        max_seq_len,
        random_beatmap=True,
        random_crop=True,
    ):
        self.audio_ids = audio_ids
        self.audio_to_samples = audio_to_samples
        self.beatmap_shards = beatmap_shards
        self.audio_embeddings = audio_embeddings
        self.max_seq_len = max_seq_len
        self.random_beatmap = random_beatmap
        self.random_crop = random_crop

    def __len__(self):
        return len(self.audio_ids)

    def __getitem__(self, idx):
        audio_id = self.audio_ids[idx]
        locations = self.audio_to_samples[audio_id]

        if self.random_beatmap:
            location = random.choice(locations)
        else:
            location = locations[0]

        sample = self.beatmap_shards[
            location["shard"]
        ]["samples"][location["offset"]]

        events = sample["events"].long()

        # 长谱训练时随机取一个连续片段。
        # 完整 embedding 提取时会对所有 chunk 取平均，见后面的代码。
        if events.shape[0] > self.max_seq_len:
            if self.random_crop:
                start = random.randint(
                    0,
                    events.shape[0] - self.max_seq_len,
                )
            else:
                start = 0

            events = events[start:start + self.max_seq_len]

        return {
            "events": events,
            "audio_embedding": self.audio_embeddings[audio_id],
            "star_rating": float(sample["star_rating"]),
            "audio_id": audio_id,
            "beatmap_id": sample["beatmap_id"],
        }


def collate_batch(batch):
    max_len = max(x["events"].shape[0] for x in batch)

    # 0 是合法 delta，所以必须同时提供 padding mask
    events = torch.zeros(
        len(batch),
        max_len,
        3,
        dtype=torch.long,
    )

    padding_mask = torch.ones(
        len(batch),
        max_len,
        dtype=torch.bool,
    )

    for i, item in enumerate(batch):
        length = item["events"].shape[0]
        events[i, :length] = item["events"]
        padding_mask[i, :length] = False

    return {
        "events": events,
        "padding_mask": padding_mask,
        "audio_embeddings": torch.stack(
            [x["audio_embedding"] for x in batch]
        ),
        "star_ratings": torch.tensor(
            [x["star_rating"] for x in batch],
            dtype=torch.float32,
        ),
    }

In [ ]:
# =========================
# 6. Beatmap Transformer encoder
# =========================

class BeatmapTransformerEncoder(nn.Module):
    def __init__(
        self,
        d_model=256,
        audio_dim=768,
        max_seq_len=1024,
        max_delta=768,
        num_heads=8,
        num_layers=4,
        ff_dim=1024,
        dropout=0.1,
    ):
        super().__init__()

        self.max_seq_len = max_seq_len
        self.max_delta = max_delta

        self.delta_embedding = nn.Embedding(max_delta + 1, d_model)
        self.type_embedding = nn.Embedding(3, d_model)
        self.lane_embedding = nn.Embedding(4, d_model)
        self.position_embedding = nn.Embedding(max_seq_len + 1, d_model)

        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.transformer = nn.TransformerEncoder(
            layer,
            num_layers=num_layers,
            norm=nn.LayerNorm(d_model),
        )

        # 输出为与 audio embedding 相同的 768 维
        self.beatmap_projection = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, audio_dim),
        )

        # 辅助头：只从 event sequence 预测星级
        self.star_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, 1),
        )

    def forward(self, events, padding_mask):
        """
        events: [B, L, 3]
        padding_mask: [B, L], True 表示 padding
        """
        batch_size, seq_len, _ = events.shape

        delta = events[:, :, 0].clamp(0, self.max_delta)
        event_type = events[:, :, 1].clamp(0, 2)
        lane = events[:, :, 2].clamp(0, 3)

        positions = torch.arange(
            1,
            seq_len + 1,
            device=events.device,
        ).unsqueeze(0)

        x = (
            self.delta_embedding(delta)
            + self.type_embedding(event_type)
            + self.lane_embedding(lane)
            + self.position_embedding(positions)
        )

        cls = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls, x], dim=1)

        cls_padding = torch.zeros(
            batch_size,
            1,
            dtype=torch.bool,
            device=events.device,
        )
        full_padding_mask = torch.cat(
            [cls_padding, padding_mask],
            dim=1,
        )

        x = self.transformer(
            x,
            src_key_padding_mask=full_padding_mask,
        )

        chart_feature = x[:, 0]  # [B, D_MODEL]
        beatmap_embedding = F.normalize(
            self.beatmap_projection(chart_feature),
            dim=-1,
        )

        predicted_star = self.star_head(chart_feature).squeeze(-1)

        return chart_feature, beatmap_embedding, predicted_star

In [ ]:
# =========================
# 7. 训练
# =========================

train_dataset = AudioGroupedBeatmapDataset(
    train_audio_ids,
    audio_to_samples,
    beatmap_shards,
    audio_embeddings,
    max_seq_len=MAX_SEQ_LEN,
    random_beatmap=True,
    random_crop=True,
)

val_dataset = AudioGroupedBeatmapDataset(
    val_audio_ids,
    audio_to_samples,
    beatmap_shards,
    audio_embeddings,
    max_seq_len=MAX_SEQ_LEN,
    random_beatmap=False,
    random_crop=False,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_batch,
    pin_memory=(DEVICE == "cuda"),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_batch,
    pin_memory=(DEVICE == "cuda"),
)

model = BeatmapTransformerEncoder(
    d_model=D_MODEL,
    audio_dim=AUDIO_DIM,
    max_seq_len=MAX_SEQ_LEN,
    max_delta=768,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    ff_dim=FF_DIM,
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scaler = torch.cuda.amp.GradScaler(
    enabled=(DEVICE == "cuda")
)


def compute_loss(batch):
    events = batch["events"].to(DEVICE, non_blocking=True)
    padding_mask = batch["padding_mask"].to(
        DEVICE,
        non_blocking=True,
    )

    # 音频 embedding 不训练，仅作为固定 target
    audio = batch["audio_embeddings"].to(
        DEVICE,
        non_blocking=True,
    )
    audio = F.normalize(audio, dim=-1)

    star = batch["star_ratings"].to(
        DEVICE,
        non_blocking=True,
    )

    _, beatmap_embedding, predicted_star = model(
        events,
        padding_mask,
    )

    # 一批中每首歌只出现一次，因此标准 CLIP / InfoNCE 可直接使用
    logits = (
        beatmap_embedding @ audio.T
    ) / TEMPERATURE

    labels = torch.arange(
        logits.shape[0],
        device=DEVICE,
    )

    contrastive_loss = 0.5 * (
        F.cross_entropy(logits, labels)
        + F.cross_entropy(logits.T, labels)
    )

    star_loss = F.smooth_l1_loss(predicted_star, star)

    total_loss = contrastive_loss + LAMBDA_STAR * star_loss

    accuracy = (logits.argmax(dim=1) == labels).float().mean()

    return total_loss, contrastive_loss, star_loss, accuracy


best_val_loss = float("inf")
checkpoint_path = OUTPUT_DIR / "beatmap_transformer_best.pt"

for epoch in range(1, EPOCHS + 1):
    model.train()

    train_losses = []
    train_accs = []

    for batch in tqdm(
        train_loader,
        desc=f"Epoch {epoch}/{EPOCHS} train",
        leave=False,
    ):
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(
            enabled=(DEVICE == "cuda")
        ):
            total_loss, _, _, accuracy = compute_loss(batch)

        scaler.scale(total_loss).backward()
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0,
        )

        scaler.step(optimizer)
        scaler.update()

        train_losses.append(total_loss.item())
        train_accs.append(accuracy.item())

    model.eval()

    val_losses = []
    val_accs = []

    with torch.no_grad():
        for batch in val_loader:
            total_loss, _, _, accuracy = compute_loss(batch)

            val_losses.append(total_loss.item())
            val_accs.append(accuracy.item())

    train_loss = float(np.mean(train_losses))
    train_acc = float(np.mean(train_accs))
    val_loss = float(np.mean(val_losses))
    val_acc = float(np.mean(val_accs))

    print(
        f"Epoch {epoch:02d} | "
        f"train loss {train_loss:.4f}, acc {train_acc:.4f} | "
        f"val loss {val_loss:.4f}, acc {val_acc:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "config": {
                    "audio_dim": AUDIO_DIM,
                    "d_model": D_MODEL,
                    "max_seq_len": MAX_SEQ_LEN,
                    "max_delta": 768,
                    "num_heads": NUM_HEADS,
                    "num_layers": NUM_LAYERS,
                    "ff_dim": FF_DIM,
                },
                "best_val_loss": best_val_loss,
                "epoch": epoch,
            },
            checkpoint_path,
        )

print("Best checkpoint:", checkpoint_path)

/tmp/ipykernel_743/2338009208.py:39: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(
/tmp/ipykernel_743/915110977.py:59: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(


Epoch 1/40 train:   0%|          | 0/175 [00:00<?, ?it/s]

/tmp/ipykernel_743/915110977.py:128: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(


Epoch 01 | train loss 3.3362, acc 0.0795 | val loss 3.2688, acc 0.0849


Epoch 2/40 train:   0%|          | 0/175 [00:00<?, ?it/s]

...

Epoch 38 | train loss 2.4801, acc 0.3512 | val loss 2.8428, acc 0.1973


Epoch 39/40 train:   0%|          | 0/175 [00:00<?, ?it/s]

Epoch 39 | train loss 2.4636, acc 0.3576 | val loss 2.8110, acc 0.2042


Epoch 40/40 train:   0%|          | 0/175 [00:00<?, ?it/s]

Epoch 40 | train loss 2.4511, acc 0.3689 | val loss 2.7921, acc 0.2203
Best checkpoint: /content/drive/MyDrive/osu_mania_dataset/beatmap_embeddings/beatmap_transformer_best.pt


In [ ]:
# =========================
# 8. 用最佳模型提取所有完整 beatmap embeddings
# =========================

checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE,
    weights_only=False,
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()


@torch.no_grad()
def encode_full_beatmap(events, chunk_size=32):
    """
    长谱面切成多个长度最多 MAX_SEQ_LEN 的连续 chunk，
    分别编码后取平均，得到整张谱面 embedding。
    """
    events = events.long()

    chunks = [
        events[i:i + MAX_SEQ_LEN]
        for i in range(0, events.shape[0], MAX_SEQ_LEN)
    ]

    chunk_embeddings = []

    for start in range(0, len(chunks), chunk_size):
        batch_chunks = chunks[start:start + chunk_size]

        max_len = max(x.shape[0] for x in batch_chunks)

        batch_events = torch.zeros(
            len(batch_chunks),
            max_len,
            3,
            dtype=torch.long,
        )

        batch_mask = torch.ones(
            len(batch_chunks),
            max_len,
            dtype=torch.bool,
        )

        for i, chunk in enumerate(batch_chunks):
            length = chunk.shape[0]
            batch_events[i, :length] = chunk
            batch_mask[i, :length] = False

        _, embeddings, _ = model(
            batch_events.to(DEVICE),
            batch_mask.to(DEVICE),
        )

        chunk_embeddings.append(embeddings.cpu())

    # 每个 chunk 的 embedding 已归一化；平均后再归一化
    full_embedding = torch.cat(
        chunk_embeddings,
        dim=0,
    ).mean(dim=0)

    return F.normalize(full_embedding, dim=0)

In [ ]:
# =========================
# 9. 保存 768 维 beatmap embeddings
# =========================

from collections import defaultdict

embedding_index = {
    "model_checkpoint": str(checkpoint_path),
    "embedding_dim": AUDIO_DIM,
    "source_beatmap_event_dir": str(BEATMAP_EVENT_DIR),
    "beatmap_id_to_embedding": {},
    "audio_to_beatmaps": defaultdict(list),
}

for shard_id in tqdm(
    sorted(beatmap_shards),
    desc="Extracting beatmap embeddings",
):
    source_shard = beatmap_shards[shard_id]

    output_samples = []

    for source_offset, sample in enumerate(source_shard["samples"]):
        embedding = encode_full_beatmap(sample["events"])

        output_offset = len(output_samples)

        output_samples.append({
            "beatmap_id": sample["beatmap_id"],
            "audio_id": sample["audio_id"],
            "difficulty_name": sample["difficulty_name"],
            "star_rating": sample["star_rating"],
            "embedding": embedding.to(torch.float32),
        })

        embedding_index["beatmap_id_to_embedding"][
            sample["beatmap_id"]
        ] = {
            "shard": shard_id,
            "offset": output_offset,
        }

        embedding_index["audio_to_beatmaps"][
            sample["audio_id"]
        ].append(sample["beatmap_id"])

    output_path = OUTPUT_DIR / f"embedding_shard_{shard_id:05d}.pt"

    torch.save(
        {
            "source_shard_id": shard_id,
            "embedding_dim": AUDIO_DIM,
            "samples": output_samples,
        },
        output_path,
    )

embedding_index["audio_to_beatmaps"] = dict(
    embedding_index["audio_to_beatmaps"]
)

with open(
    OUTPUT_DIR / "beatmap_embedding_index.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        embedding_index,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("Finished.")
print("Embedding folder:", OUTPUT_DIR)

Extracting beatmap embeddings:   0%|          | 0/38 [00:00<?, ?it/s]

Finished.
Embedding folder: /content/drive/MyDrive/osu_mania_dataset/beatmap_embeddings
